# conv-stride-downsample — worked example 1: Predict the H and W of a strided 2-D conv

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-stride-downsample`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A 2-D convolution with stride `S` and no padding shrinks each spatial dim by the formula `out = (in - K) // S + 1`. The `+ 1` is the leading window at index 0; the floor division counts how many further stride-`S` jumps fit before the kernel runs off the edge. The same formula applies independently to height and width.

## Worked solution

**Goal.** Given an input of shape `(N, C, H_in, W_in)`, kernel `K`, and stride `S`, predict the output spatial size, then confirm with `F.conv2d`.

**Step 1 — apply the formula per axis.** Because height and width are processed independently by a square kernel, we compute `H_out = (H_in - K) // S + 1` and `W_out = (W_in - K) // S + 1` separately. With `H_in=32, W_in=20, K=5, S=3`: `H_out = (32-5)//3 + 1 = 27//3 + 1 = 9 + 1 = 10` and `W_out = (20-5)//3 + 1 = 15//3 + 1 = 5 + 1 = 6`.

**Step 2 — why floor division, not exact.** Windows start at positions `0, S, 2S, ...`. The last valid start is the largest multiple of `S` that is `<= in - K`. The count of such starts including 0 is exactly `(in - K) // S + 1`. Any leftover columns that cannot host a full kernel are simply dropped — that is what the floor discards.

**Step 3 — verify against the real op.** We build a random `(N, C_in, H_in, W_in)` tensor and a `Conv2d` with `C_out` filters. The channel dim of the output equals `C_out` (unaffected by stride); only the spatial dims follow our arithmetic. Comparing `tuple(y.shape[2:])` to our predicted `(H_out, W_out)` proves the formula.

In [ ]:
def strided_conv2d_outshape(h_in, w_in, k, s):
    h_out = (h_in - k) // s + 1
    w_out = (w_in - k) // s + 1
    return h_out, w_out

# exercise it against a real conv2d
t.manual_seed(0)
N, C_in, C_out = 2, 3, 7
h_in, w_in, k, s = 32, 20, 5, 3
x = t.randn(N, C_in, h_in, w_in)
conv = t.nn.Conv2d(C_in, C_out, kernel_size=k, stride=s)
y = conv(x)
pred = strided_conv2d_outshape(h_in, w_in, k, s)
print('predicted HxW:', pred)
print('actual   HxW:', tuple(y.shape[2:]))
print('channels out:', y.shape[1])